In [1]:
import os
import numpy as np
import pandas as pd
import pickle
import spacy

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.tag import pos_tag
from nltk.stem import WordNetLemmatizer
from nltk.probability import FreqDist
from nltk.classify import NaiveBayesClassifier, accuracy

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
DATASET_PATH = './jobpostingdata.csv'  # 1 = penipuan
MODEL_PATH = './naive_bayes_model.pkl'

stop_words = stopwords.words('english')
lemmatizer = WordNetLemmatizer()
vectorizer = TfidfVectorizer()

In [3]:
def load_dataset():
    df = pd.read_csv(DATASET_PATH)
    return df['title'], df['fraudulent'], df['text']

def get_tag(tag):
    if tag.startswith('J'):
        return 'a'
    elif tag.startswith('R'):
        return 'r'
    elif tag.startswith('V'):
        return 'v'
    else:
        return 'n'
    
def preprocess(text):
    text = text.lower()
    text = word_tokenize(text)
    text = [w for w in text if w not in stop_words]
    text = [w for w in text if w.isalpha()]
    
    tagged = pos_tag(text)
    text = [lemmatizer.lemmatize(w, get_tag(t)) for (w, t) in tagged]
    return ' '.join(text)

def extract_feature(freq_dist, text):
    words = text.split()
    return {word: (word in words) for word in freq_dist}

def train(text, labels):
    texts = text.apply(preprocess)
    all_words = ' '.join(texts).split()
    
    freq_dist = FreqDist(all_words)
    feature_set = [(extract_feature(freq_dist, text), label)
                   for (text, label) in zip(texts, labels)]
    split = int(0.8 * len(feature_set))
    train_set = feature_set[:split]
    test_set = feature_set[split:]
    
    classifier = NaiveBayesClassifier.train(train_set)
    print(f"Accuracy: {accuracy(classifier, test_set)}")
    
    vectors = vectorizer.fit_transform(texts)  # FIX: was fit_transform([texts]) (1 giant doc) + a redundant call
    
    return classifier, freq_dist, vectorizer, vectors

def train_or_load_model():
    if not os.path.exists(MODEL_PATH):
        titles, labels, texts = load_dataset()
        model = train(texts, labels)
        
        with open(MODEL_PATH, 'wb') as f:
            pickle.dump((*model, titles, texts), f)
        return (*model, titles, texts)
    else:
        with open(MODEL_PATH, 'rb') as f:
            return pickle.load(f)  # FIX: removed duplicate pickle.load(f) above this (caused EOFError) + 'return pickle.load' returned the function

In [4]:
def menu():
    text = ''
    text_category = ''
    
    classifier, freq_dist, vectorizer, vectors, titles, texts = train_or_load_model()
    
    while True:
        print('Job Posting Classification')
        print('Your Text:', text if text else 'None')
        print('Your Text Category:', text_category if text_category else 'None')  # FIX: now prints the category value, not just the label
        print('1. Write your texts')  # FIX: typo 'yout' -> 'your'
        print('2. View Recommendation')
        print('3. View NER')
        print('4. Exit')
        
        choice = input('>> ')
        if choice == '1':
            text = input('Write your text: ')  # FIX: store input into `text` (was preprocessing the old empty `text`)
            if len(text) < 20 or len(text.split()) < 3:
                text = ''
                continue
            clean_text = preprocess(text)
            feature = extract_feature(freq_dist, clean_text)
            result = classifier.classify(feature)
            if result == 1:
                text_category = 'Fake Job Posting'
            else:
                text_category = 'Real Job Posting'
            
        elif choice == '2':
            if len(text) == 0:
                print('No Text')
                continue
            clean_text = preprocess(text)
            vector = vectorizer.transform([clean_text])  # FIX: wrap in list (was transform(text) -> iterated over characters)
            similar = cosine_similarity(vector, vectors)[0]
            top = np.argsort(similar)[-5:][::-1]
            
            for i in top:
                print('Title:', titles.iloc[i])
                print('Similarity:', similar[i])
                
        elif choice == '3':
            if len(text) == 0:
                print('No Text')
                continue
            nlp = spacy.load('en_core_web_sm')
            doc = nlp(text)
            
            # FIX: removed PERSON-only filter (spec: show ALL entities) and actually print them
            categories = {}
            for ent in doc.ents:
                categories.setdefault(ent.label_, []).append(ent.text)
            
            if not categories:
                print('No named entities found')
            for label, ents in categories.items():
                print(f'{label}: {", ".join(ents)}')  # FIX: results are now printed (were silently discarded)
            
        elif choice == '4':
            break  # FIX: break to terminate (was `pass`, which looped forever)
        else:
            print('Invalid choice!')

menu()

Job Posting Classification
Your Text: None
Your Text Category: None
1. Write your texts
2. View Recommendation
3. View NER
4. Exit
Invalid choice!
Job Posting Classification
Your Text: None
Your Text Category: None
1. Write your texts
2. View Recommendation
3. View NER
4. Exit
Job Posting Classification
Your Text: Joko makan nasi goreng bareng anwar dan siti di kantin binus\
Your Text Category: Fake Job Posting
1. Write your texts
2. View Recommendation
3. View NER
4. Exit
